# 03 | Statistical Analysis — Ontario PHO Surveillance

**Purpose:** estimate descriptive time trends in reported disease cases, compare observed vs fitted counts, and quantify variation in reported outcomes.  
**Inputs:** three cleaned CSV files from `outputs/processed/` generated by Notebook 02.  
**Outputs:** result tables in `outputs/statistical/`.  

The analysis is at **aggregate surveillance level**, not patient level. Results are associations, not causal effects. Reported-case counts depend on testing and surveillance practices. 2025 is excluded from the default inferential analysis pending verification of its completeness.

**Tools:** pandas, numpy, scipy, statsmodels, plotly. SAS equivalents are shown as *unexecuted reference code* in Markdown only.

## 01 | Setup and locate project

In [12]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from scipy.stats import chi2
from IPython.display import display

# Works when launched from the project root OR its notebooks/ folder.
roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in roots if (p / "outputs" / "processed" / "pho_overall_tidy.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find outputs/processed/pho_overall_tidy.csv. Open this notebook inside the PHO project.")
INPUT_DIR = PROJECT_ROOT / "outputs" / "processed"
RESULT_DIR = PROJECT_ROOT / "outputs" / "statistical"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Outputs:", RESULT_DIR)

Project: c:\Users\Honar\Downloads\PHO Project
Outputs: c:\Users\Honar\Downloads\PHO Project\outputs\statistical


## 02 | Load and validate the three processed datasets

In [13]:
overall = pd.read_csv(INPUT_DIR / "pho_overall_tidy.csv")
age_sex = pd.read_csv(INPUT_DIR / "pho_age_sex_tidy.csv")
outcomes = pd.read_csv(INPUT_DIR / "pho_outcomes_tidy.csv")
required = {
    "overall": (overall, ["Public health unit", "Disease", "Year", "Cases", "Population", "Rate per 100,000"]),
    "age_sex": (age_sex, ["Public health unit", "Disease", "Year", "Sex", "Age group", "Number of cases"]),
    "outcomes": (outcomes, ["Public health unit", "Disease", "Year", "Outcome", "Cases with outcome", "Total cases", "Percent of cases with outcome"]),
}
for name, (df, cols) in required.items():
    missing = sorted(set(cols) - set(df.columns))
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    print(f"{name}: {len(df):,} rows; years {int(df['Year'].min())}–{int(df['Year'].max())}")
for df, numeric_cols in [(overall, ["Cases", "Population", "Rate per 100,000"]),
                         (age_sex, ["Number of cases"]),
                         (outcomes, ["Cases with outcome", "Total cases", "Percent of cases with outcome"])]:
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
print("Duplicate overall grain:", overall.duplicated(["Public health unit", "Disease", "Year"]).sum())
print("Duplicate outcome grain:", outcomes.duplicated(["Public health unit", "Disease", "Year", "Outcome"]).sum())

overall: 20,034 rows; years 2016–2025
age_sex: 213,741 rows; years 2016–2025
outcomes: 648 rows; years 2016–2025
Duplicate overall grain: 0
Duplicate outcome grain: 0


## 03 | Set analysis scope

Select one Ontario-level disease with sufficient valid annual counts and population denominators. **Do not sum Ontario and PHU rows together**: that would double-count. `END_YEAR=2024` deliberately omits potentially provisional 2025 data. Change it only after checking the PHO release notes.

In [14]:
DISEASE = "Lyme Disease"
START_YEAR, END_YEAR = 2016, 2024

ont = overall.loc[
    overall["Public health unit"].eq("Ontario")
    & overall["Disease"].eq(DISEASE)
    & overall["Year"].between(START_YEAR, END_YEAR),
    ["Year", "Cases", "Population", "Rate per 100,000"]
].copy().sort_values("Year")
ont = ont.dropna(subset=["Year", "Cases", "Population"])
ont = ont.loc[(ont["Cases"] >= 0) & (ont["Population"] > 0)]
if ont["Year"].duplicated().any():
    raise ValueError("Multiple Ontario rows for a disease-year; inspect data grain.")
if len(ont) < 5:
    raise ValueError(f"Only {len(ont)} usable years for {DISEASE}; choose a disease with >=5.")
ont["Year_centered"] = ont["Year"] - ont["Year"].min()
ont["Observed_rate"] = 1e5 * ont["Cases"] / ont["Population"]
display(ont[["Year", "Cases", "Population", "Observed_rate"]])

,Year,Cases,Population,Observed_rate
32,2016,402,13876500,2.896984
1957,2017,1045,14078499,7.422666
3884,2018,652,14326746,4.550929
5869,2019,1192,14573565,8.179193
7854,2020,884,14761811,5.988425
9839,2021,1764,14842488,11.884800
11824,2022,1524,15155836,10.055532
13841,2023,1903,15632481,12.173372
15917,2024,2402,16144797,14.877858


## 04 | Poisson count regression with population offset

Model: `log(E[Cases]) = intercept + beta × years_since_start + log(Population)`.  
`exp(beta)` is the estimated multiplicative annual rate ratio (RR). Check overdispersion; Poisson intervals can be too narrow when variance exceeds mean. Sparse time series and serial correlation also limit conventional inference.

In [15]:
X = sm.add_constant(ont[["Year_centered"]], has_constant="add")
poisson = sm.GLM(ont["Cases"], X, family=sm.families.Poisson(), offset=np.log(ont["Population"])).fit()
phi = poisson.pearson_chi2 / poisson.df_resid if poisson.df_resid > 0 else np.nan
# Scale-adjusted (quasi-Poisson-style) SEs; descriptive sensitivity, not serial-correlation correction.
fit = sm.GLM(ont["Cases"], X, family=sm.families.Poisson(), offset=np.log(ont["Population"])).fit(scale=max(1.0, phi))
beta = fit.params["Year_centered"]
ci = fit.conf_int().loc["Year_centered"]
trend = pd.DataFrame([{
    "Disease": DISEASE, "Years": f"{int(ont.Year.min())}–{int(ont.Year.max())}",
    "Annual rate ratio": np.exp(beta), "RR lower 95%": np.exp(ci.iloc[0]),
    "RR upper 95%": np.exp(ci.iloc[1]), "P value (approx)": fit.pvalues["Year_centered"],
    "Pearson dispersion": phi, "Number of years": len(ont)
}])
display(trend.round(4))
if phi > 1.5:
    print("CAUTION: Evidence of overdispersion. Treat p-values/intervals cautiously and investigate alternate models.")
ont["Fitted_cases"] = fit.predict(X, offset=np.log(ont["Population"]))
ont["Fitted_rate"] = ont["Fitted_cases"] / ont["Population"] * 100_000

,Disease,Years,Annual rate ratio,RR lower 95%,RR upper 95%,P value (approx),Pearson dispersion,Number of years
0,Lyme Disease,2016–2024,1.1643,1.0944,1.2386,0.0,70.1974,9


CAUTION: Evidence of overdispersion. Treat p-values/intervals cautiously and investigate alternate models.


### SAS equivalent (reference only, **not executed or validated**)

The SAS input must be the **same filtered Ontario disease-year data**, with `Year_centered`, `Cases`, and strictly positive `Population`. `SCALE=PEARSON` estimates a dispersion adjustment; output may differ slightly from the Python `max(1, phi)` sensitivity calculation.

```sas
/* Run only after importing the filtered Ontario disease-year table as ont */
data ont_model;
  set ont;
  log_population = log(Population);
run;
proc genmod data=ont_model;
  model Cases = Year_centered / dist=poisson link=log offset=log_population scale=pearson;
  estimate 'Annual rate ratio' Year_centered 1 / exp;
run;
```

## 05 | Plot observed and fitted annual rates

In [16]:
plot_df = ont.melt(id_vars="Year", value_vars=["Observed_rate", "Fitted_rate"],
                   var_name="Series", value_name="Plot_rate")
fig = px.line(plot_df, x="Year", y="Plot_rate", color="Series", markers=True,
              title=f"{DISEASE}: observed vs modeled rate · Ontario")
fig.update_xaxes(dtick=1)
fig.update_yaxes(title="Rate per 100,000 population")
fig.show()

## 06 | Largest year-over-year changes and unusual residuals

This is **screening**, not proof of an outbreak or a statistically validated anomaly. A standardized Pearson residual larger than ±2 gets flagged as a review candidate; the threshold is heuristic and ignores multiple comparisons.

In [17]:
ont["YoY_change_cases"] = ont["Cases"].diff()
ont["Pearson_residual"] = (ont["Cases"] - ont["Fitted_cases"]) / np.sqrt(ont["Fitted_cases"].clip(lower=1))
ont["Review_flag"] = ont["Pearson_residual"].abs() > 2
screen = ont[["Year", "Cases", "YoY_change_cases", "Fitted_cases", "Pearson_residual", "Review_flag"]]
display(screen.round(2))
# Many flags can indicate overdispersion or model misspecification, not confirmed outbreaks.
print(f"Screening flags: {ont['Review_flag'].sum()} of {len(ont)} years; interpret cautiously.")


,Year,Cases,YoY_change_cases,Fitted_cases,Pearson_residual,Review_flag
32,2016,402,NaN,606.60,-8.31,True
1957,2017,1045,643.0,716.56,12.27,True
3884,2018,652,-393.0,849.01,-6.76,True
5869,2019,1192,540.0,1005.54,5.88,True
7854,2020,884,-308.0,1185.88,-8.77,True
9839,2021,1764,880.0,1388.28,10.08,True
11824,2022,1524,-240.0,1650.51,-3.11,True
13841,2023,1903,379.0,1982.15,-1.78,False
15917,2024,2402,499.0,2383.47,0.38,False


Screening flags: 7 of 9 years; interpret cautiously.


## 07 | Age and sex: descriptive distributions only

Age bands can differ by disease. Counts below are **not age-standardized** and do not imply comparative individual risk. Avoid aggregating overlapping age bands or mixing Ontario totals with local PHU strata.

In [18]:
demo = age_sex.loc[
    age_sex["Public health unit"].eq("Ontario")
    & age_sex["Disease"].eq(DISEASE)
    & age_sex["Year"].eq(END_YEAR)
    & age_sex["Sex"].isin(["Female", "Male"]),
    ["Age group", "Sex", "Number of cases"]
].dropna(subset=["Age group", "Number of cases"])
if demo.empty:
    print("No Ontario age/sex rows for selected disease and end year.")
else:
    demo_plot = demo.groupby(["Age group", "Sex"], as_index=False)["Number of cases"].sum()
    display(demo_plot.head(15))
    fig = px.bar(demo_plot, x="Age group", y="Number of cases", color="Sex", barmode="group",
                 title=f"Reported cases by age and sex · {DISEASE} · Ontario · {END_YEAR}")
    age_order = ["0-4", "5-9", "10-19", "20-29", "30-39",
                 "40-49", "50-59", "60-69", "70+", "Unknown"]
    fig.update_xaxes(categoryorder="array", categoryarray=age_order)
    fig.show()

,Age group,Sex,Number of cases
0,0-4,Female,33
1,0-4,Male,17
2,10-19,Female,69
3,10-19,Male,90
4,20-29,Female,49
5,20-29,Male,97
6,30-39,Female,89
7,30-39,Male,166
8,40-49,Female,111
9,40-49,Male,197


## 08 | Outcomes: hospitalization and deaths, independent scale

Outcome records are from a **separate Ontario-only source**. Show only provided disease/outcome combinations; missing combinations are not zero. Percentages are descriptive proportions among reported cases.

In [19]:
OUTCOME_DISEASE = "Salmonellosis"
out = outcomes.loc[
    outcomes["Public health unit"].eq("Ontario")
    & outcomes["Disease"].eq(OUTCOME_DISEASE)
    & outcomes["Year"].between(START_YEAR, END_YEAR)
    & outcomes["Outcome"].isin(["Hospitalizations", "Deaths"])
].copy()
if out.empty:
    print("No selected outcome data available.")
else:
    from plotly.subplots import make_subplots
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Hospitalizations", "Deaths"), vertical_spacing=.15)
    for i, label in enumerate(["Hospitalizations", "Deaths"], start=1):
        part = out.loc[out["Outcome"].eq(label)].sort_values("Year")
        fig.add_scatter(x=part["Year"], y=part["Percent of cases with outcome"],
                        mode="lines+markers", name=label, row=i, col=1)
        fig.update_yaxes(title_text="Percent (%)", rangemode="tozero", row=i, col=1)
    fig.update_layout(height=650, showlegend=False, title=f"Reported outcomes · {OUTCOME_DISEASE} · Ontario")
    fig.update_xaxes(title_text="Year", row=2, col=1, dtick=1)
    fig.show()
    display(out[["Year", "Outcome", "Cases with outcome", "Total cases", "Percent of cases with outcome"]].head(12))

,Year,Outcome,Cases with outcome,Total cases,Percent of cases with outcome
478,2016,Deaths,7.0,3104.0,0.225515
479,2017,Deaths,11.0,2739.0,0.401606
480,2018,Deaths,8.0,2677.0,0.298842
481,2019,Deaths,7.0,2388.0,0.293132
482,2020,Deaths,5.0,1516.0,0.329815
483,2021,Deaths,3.0,1197.0,0.250627
484,2022,Deaths,9.0,1781.0,0.505334
485,2023,Deaths,8.0,2386.0,0.335289
486,2024,Deaths,10.0,2616.0,0.382263
488,2016,Hospitalizations,337.0,3104.0,10.856959


### SAS equivalent for outcome summaries (reference only, **not executed or validated**)

```sas
/* Same Ontario outcome extract as 'out'; this is DESCRIPTIVE, not a model */
proc sort data=out; by Outcome Year; run;
proc print data=out noobs;
  var Year Outcome Cases_with_outcome Total_cases Percent_of_cases_with_outcome;
run;
```

In SAS, rename spaced column headers consistently before executing this example.

## 09 | Save compact results and reproducibility notes

In [20]:
trend.to_csv(RESULT_DIR / "annual_trend_summary.csv", index=False)
screen.to_csv(RESULT_DIR / "annual_trend_diagnostics.csv", index=False)
print("Saved:", RESULT_DIR / "annual_trend_summary.csv")
print("Saved:", RESULT_DIR / "annual_trend_diagnostics.csv")

Saved: c:\Users\Honar\Downloads\PHO Project\outputs\statistical\annual_trend_summary.csv
Saved: c:\Users\Honar\Downloads\PHO Project\outputs\statistical\annual_trend_diagnostics.csv


## 10 | Interpretation and limitations

- **Annual rate ratio:** >1 corresponds to an increasing modeled rate; <1 corresponds to a decreasing modeled rate. The model assumes a log-linear rate trajectory across the selected years.
- **Inference caveat:** a small series of annual surveillance totals is subject to temporal correlation, outliers, definition changes, reporting delays and pandemic-era disruptions. A conventional GLM p-value should not be described as definitive evidence.
- **Dispersion:** the Python fit adjusts standard errors upward for observed overdispersion but does **not** solve time dependence; explore alternative trend approaches or segmented models if scientifically justified.
- **Geography:** never sum Ontario and PHU records. Comparisons across PHUs require appropriate denominators and contextual interpretation.
- **Demographics:** age/sex counts are unadjusted. Do not infer individual risk or compare overlapping age bands as independent categories.
- **Outcomes:** available for only selected diseases, Ontario-only; reported outcome percentages do not imply population-wide severity or case fatality.
- **Data release:** check 2025 provisional/revised status before inclusion. Document PHO attribution and redistribution conditions before publishing processed CSVs.
- **SAS:** Markdown examples are illustrative and not run, tested, or validated; do not advertise SAS equivalence as independently verified.

**Suggested README description:** “Python statistical analysis of PHO aggregate Ontario surveillance data, with illustrative SAS syntax alongside selected methods. Inference is exploratory, and models are sensitive to reporting processes and time-series assumptions.”